# 01 — Análisis Exploratorio de Datos (EDA)
**TP Semana 6 — Arquitecturas CNN sobre Blood Cell Image Dataset**

Este notebook realiza el análisis exploratorio del dataset *Blood Cell Images* de Kaggle
(`paultimothymooney/blood-cells`). Cubre:

1. Descarga y descompresión del dataset desde Kaggle.
2. Inventario de archivos por split (`TRAIN` / `TEST`) y por clase.
3. Distribución de clases (balance / desbalance).
4. Inspección visual de muestras por clase.
5. Análisis de propiedades de las imágenes (tamaño, canales, estadísticos RGB).
6. Conclusiones para la etapa de modelado.

**Clases:** `EOSINOPHIL`, `LYMPHOCYTE`, `MONOCYTE`, `NEUTROPHIL`.


## 1. Configuración del entorno

In [ ]:
# Reproducibilidad
import os, random, sys
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print("Python:", sys.version.split()[0])
print("NumPy :", np.__version__)


In [ ]:
# Librerías base
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from pathlib import Path

%matplotlib inline
plt.rcParams["figure.dpi"] = 110


## 2. Descarga del dataset desde Kaggle

> En Google Colab, sube tu archivo `kaggle.json` (Account → Create New API Token) al ejecutar la celda siguiente.
> Si ya descargaste el dataset manualmente y lo subiste a Colab/Drive, salta a la celda *Ruta local*.


In [ ]:
# --- Opción A: descarga automática vía Kaggle API ---
# Descomenta este bloque la primera vez que ejecutes el notebook en Colab.

# from google.colab import files
# files.upload()  # selecciona kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip -q install kaggle
# !kaggle datasets download -d paultimothymooney/blood-cells -p /content/data --unzip

# --- Opción B: ruta local (si ya descargaste el dataset) ---
DATA_ROOT = Path("/content/data")
if not DATA_ROOT.exists():
    DATA_ROOT = Path("./data")

print("DATA_ROOT:", DATA_ROOT, "| existe:", DATA_ROOT.exists())


In [ ]:
# El zip de Kaggle expande dos carpetas: 'dataset-master' (pocas imgs) y
# 'dataset2-master' (las ~12,500 imgs con split TRAIN/TEST). Usamos la 2.
CANDIDATES = [
    DATA_ROOT / "dataset2-master" / "dataset2-master" / "images",
    DATA_ROOT / "dataset2-master" / "images",
    DATA_ROOT / "images",
]
IMAGES_ROOT = next((p for p in CANDIDATES if p.exists()), None)
assert IMAGES_ROOT is not None, (
    "No se encontró el directorio de imágenes. "
    "Verifica que el dataset esté descomprimido en DATA_ROOT."
)
print("IMAGES_ROOT:", IMAGES_ROOT)

TRAIN_DIR = IMAGES_ROOT / "TRAIN"
TEST_DIR  = IMAGES_ROOT / "TEST"
print("TRAIN:", TRAIN_DIR.exists(), "| TEST:", TEST_DIR.exists())


## 3. Inventario por clase y split

In [ ]:
CLASSES = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
print("Clases detectadas:", CLASSES)

def count_images(root):
    counts = {}
    for c in CLASSES:
        cdir = root / c
        if cdir.exists():
            counts[c] = sum(1 for p in cdir.iterdir()
                            if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
        else:
            counts[c] = 0
    return counts

train_counts = count_images(TRAIN_DIR)
test_counts  = count_images(TEST_DIR)

df_counts = pd.DataFrame({"train": train_counts, "test": test_counts})
df_counts["total"] = df_counts["train"] + df_counts["test"]
df_counts.loc["TOTAL"] = df_counts.sum()
df_counts


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_counts.drop("TOTAL").plot.bar(y="train", ax=axes[0], color="#3b82f6", legend=False)
axes[0].set_title("Imágenes por clase — TRAIN")
axes[0].set_ylabel("nº imágenes")
axes[0].tick_params(axis="x", rotation=20)

df_counts.drop("TOTAL").plot.bar(y="test", ax=axes[1], color="#10b981", legend=False)
axes[1].set_title("Imágenes por clase — TEST")
axes[1].set_ylabel("nº imágenes")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig("class_balance.png", dpi=120, bbox_inches="tight")
plt.show()


**Observación.** El dataset está aproximadamente balanceado entre las 4 clases
(~2,400–2,500 imágenes por clase en TRAIN y ~620 en TEST). No será necesario aplicar
*class weighting* ni *oversampling* agresivo en el modelado.


## 4. Inspección visual de muestras

In [ ]:
def show_grid(root, n_per_class=5):
    fig, axes = plt.subplots(len(CLASSES), n_per_class,
                             figsize=(2.2 * n_per_class, 2.2 * len(CLASSES)))
    for i, c in enumerate(CLASSES):
        files = sorted((root / c).glob("*"))[:n_per_class]
        for j, f in enumerate(files):
            img = Image.open(f).convert("RGB")
            ax = axes[i, j]
            ax.imshow(img)
            ax.axis("off")
            if j == 0:
                ax.set_ylabel(c, rotation=0, ha="right", va="center", fontsize=11)
    plt.suptitle(f"Muestras — {root.name}", y=1.02, fontsize=13)
    plt.tight_layout()
    plt.savefig(f"samples_{root.name}.png", dpi=120, bbox_inches="tight")
    plt.show()

show_grid(TRAIN_DIR, n_per_class=5)


## 5. Propiedades de las imágenes

In [ ]:
# Muestreamos 300 imágenes para estimar tamaños y estadísticos rápidos
sample_paths = []
for c in CLASSES:
    files = list((TRAIN_DIR / c).glob("*.jpeg")) + list((TRAIN_DIR / c).glob("*.jpg"))
    sample_paths += files[:75]

shapes, means, stds = [], [], []
for p in sample_paths:
    img = np.asarray(Image.open(p).convert("RGB"), dtype=np.float32) / 255.0
    shapes.append(img.shape)
    means.append(img.mean(axis=(0, 1)))
    stds.append(img.std(axis=(0, 1)))

shapes_df = pd.DataFrame(shapes, columns=["H", "W", "C"])
print("Tamaño único en muestra:", shapes_df.drop_duplicates().to_dict("records"))
print()
print(f"Media RGB (sobre muestra): {np.mean(means, axis=0).round(4)}")
print(f"Desv. RGB (sobre muestra): {np.mean(stds,  axis=0).round(4)}")


In [ ]:
# Histograma de intensidades por canal (sobre la muestra)
all_pixels = []
for p in sample_paths[:60]:
    arr = np.asarray(Image.open(p).convert("RGB"), dtype=np.uint8)
    all_pixels.append(arr.reshape(-1, 3))
all_pixels = np.concatenate(all_pixels, axis=0)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#ef4444", "#22c55e", "#3b82f6"]
labels = ["R", "G", "B"]
for i in range(3):
    ax.hist(all_pixels[:, i], bins=50, alpha=0.5, color=colors[i], label=labels[i])
ax.set_title("Distribución de intensidades por canal (muestra TRAIN)")
ax.set_xlabel("intensidad [0, 255]")
ax.set_ylabel("frecuencia")
ax.legend()
plt.tight_layout()
plt.savefig("rgb_hist.png", dpi=120, bbox_inches="tight")
plt.show()


## 6. Conclusiones del EDA

- **Tamaño:** ~9,957 imágenes en TRAIN y ~2,487 en TEST, balanceadas entre las 4 clases
  (≈ 2,490 train / ≈ 620 test por clase).
- **Resolución original:** 320×240 px RGB. Para los modelos *from scratch* (LeNet y VGG-11
  reducido) re-escalaremos a **64×64** para acelerar entrenamiento en CPU/Colab.
  Para Transfer Learning con ResNet-18 / VGG-16 usaremos **224×224** (input estándar de
  ImageNet).
- **Color y morfología:** las clases se diferencian por forma, tamaño y patrones de tinción
  (HEMATOXILINA-EOSINA). El color y la textura son señales discriminativas muy fuertes —
  conviene mantener los 3 canales y aplicar normalización por canal.
- **Balance:** no se requiere *class weighting*; podemos usar `CrossEntropyLoss` estándar.
- **Augmentations razonables:** flips horizontales/verticales, pequeñas rotaciones,
  variaciones de brillo/contraste. *No* aplicar inversiones de color.
